# Setup
## Dependencies

In [ ]:
%config InlineBackend.figure_format = 'retina'
from astropy.io import fits
from astropy import units as u
from astropy.coordinates import SkyCoord
import numpy as np
import matplotlib.pyplot as plt
from astropy.wcs import WCS
from photutils.centroids import centroid_quadratic
from photutils.aperture import CircularAnnulus, CircularAperture, ApertureStats, aperture_photometry
import sqlite3
from pathlib import Path
import re

from astropy.stats import SigmaClip

import warnings
from astropy.wcs import FITSFixedWarning
from astropy.utils.exceptions import AstropyUserWarning

from astropy.wcs.utils import proj_plane_pixel_scales

warnings.filterwarnings("ignore", category=FITSFixedWarning)
warnings.filterwarnings("ignore", category=AstropyUserWarning)

In [ ]:
db_path = Path("../databases/Master_Database.db").resolve()
conn = sqlite3.connect(db_path)
nonflat_mosaic_folder = Path("../output mosaics/non-flats")

flat_mosaic_folder = Path("../output mosaics/flats")

## Dictionaries

In [ ]:
sn_coordinates = {
    # 2017
    "7hhz": ("01h44m16.910s", "+12d15m18.54s"),   # SN 2017hhz, SN Ia
    "17hn": ("13h07m39.419s", "+06d20m15.23s"),   # SN 2017hn, SN Ia
    "17ms": ("10h26m42.360s", "+36d40m49.76s"),   # SN 2017ms, SN Ia
    # 2018
    "18gv": ("08h05m34.580s", "-11d26m16.87s"),   # SN 2018gv, SN Ia
    "8kss": ("06h46m23.860s", "+43d49m27.80s"),   # SN 2018kss, SN Ia
    # 2019
    "9aik": ("10h37m38.390s", "+37d27m29.56s"),   # SN 2019aik, SN Ia
    "19np": ("10h29m21.980s", "+29d30m38.30s"),   # SN 2019np, SN Ia
    "9tgm": ("22h18m10.970s", "+45d43m10.13s"),   # SN 2019tgm, SN Ia
    # 2020
    "20se": ("03h37m48.700s", "+03d16m19.56s"),   # SN 2020se, SN Ia
    "0uec": ("21h09m48.530s", "+15d08m54.89s"),   # SN 2020uec, SN Ia
    "0vey": ("04h01m16.150s", "+31d44m03.11s"),   # SN 2020vey, SN Ia
    "0wcj": ("02h52m50.310s", "-01d13m51.71s"),   # SN 2020wcj, SN Ia
    # 2021
    "21xn": ("01h37m50.930s", "+53d39m51.91s"),   # SN 2021xn, SN Ia
    # 2024
    "4ana": ("12h58m22.578s", "+27d57m58.14s"),   # SN 2024ana, SN Ia
    "4xyn": ("02h39m23.969s", "+02d39m07.88s"),   # SN 2024xyn, SN Ia
    "24ym": ("10h11m37.390s", "+74d59m38.15s"),   # SN 2024ym, SN Ia
    # 2025
    "ahkt": ("09h25m42.441s", "+22d20m05.76s"),   # SN 2025ahkt, SN Ia
    "ahsa": ("14h35m00.210s", "+00d59m10.38s"),   # SN 2025ahsa, SN Ia
    # 2026
    "aaiv": ("22h37m05.628s", "+34d24m35.19s"),   # SN 2026aaiv, SN Ia
    "abxl": ("13h23m45.388s", "+31d33m52.17s"),   # SN 2026abxl, SN Ia, not in Master_Database yet
    "acna": ("01h48m51.954s", "+10d35m11.40s"),   # SN 2026acna, SN Ia
    "acpp": ("09h48m42.485s", "+44d04m27.95s"),   # SN 2026acpp, SN IIb
    "aeiq": ("08h55m23.301s", "-07d33m40.54s"),   # SN 2026aeiq, SN Ic
    "aeiv": ("03h16m41.828s", "+80d48m09.12s"),   # SN 2026aeiv, SN Ia-91bg-like
    "aeiz": ("00h55m17.687s", "+49d59m50.14s"),   # AT 2026aeiz, unclassified
    "aepk": ("02h09m55.997s", "+10d45m36.17s"),   # SN 2026aepk, SN Iax[02cx-like]
    "aesj": ("07h48m40.024s", "+54d36m43.42s"),   # SN 2026aesj, SN Ia
    "afnj": ("02h48m41.104s", "+14d18m22.41s"),   # SN 2026afnj, SN Ia
    "afrp": ("04h50m52.231s", "-04d53m35.15s"),   # SN 2026afrp, SN Ib
    "afsg": ("12h02m02.958s", "+29d50m40.95s"),   # SN 2026afsg, SN Ib/c, not in Master_Database yet
    "afsy": ("23h33m45.021s", "+25d47m03.53s"),   # SN 2026afsy, SN
    "afys": ("04h13m54.525s", "+29d09m15.91s"),   # AT 2026afys, unclassified
    "agua": ("05h05m54.591s", "+10d10m52.74s"),   # SN 2026agua, SN Ia
    "aguc": ("04h38m04.131s", "-10d52m19.55s"),   # SN 2026aguc, SN Ia-91T-like
    "aguv": ("07h37m04.549s", "+65d35m42.56s"),   # AT 2026aguv, unclassified
    "ahab": ("09h11m02.822s", "+13d24m57.94s"),   # AT 2026ahab, unclassified
    "ahvi": ("12h13m50.443s", "+13d25m26.24s"),   # SN 2026ahvi, SN Ia
    "aicd": ("23h58m54.253s", "+12d08m52.21s"),   # SN 2026aicd, SN Ia
    "6atb": ("14h34m27.294s", "+21d56m58.85s"),   # SN 2026atb, SN Ia
    "26gm": ("04h24m37.413s", "+33d51m43.60s"),   # SN 2026gm, SN Ia
    "26kc": ("10h22m25.076s", "+04d00m01.56s"),   # SN 2026kc, SN Ia
    "akbo": ("10h39m38.911s", "+25d19m13.52s"),   # SN 2026akbo, SN Ia
}

In [ ]:
telescope_gain = {
       "pt5m": 1.0,
       "east-16": 1.0,
       "west-14": 1.02,
       "draco2": 1.02,
       "east-14": 1.02,
       "far-east-16": 1.02,
       "far-east-20": 0.8
   }

# Functions
## get_centroid

In [ ]:
def get_centroid(image_data, wcs, obj_RA, obj_Dec):
    coord_ra_dec =  SkyCoord(obj_RA, obj_Dec)
    # in form of SkyCoord('22h37m05s', '+34d24m35.37s')

    target_x, target_y = wcs.world_to_pixel(coord_ra_dec)

    supernova_xy_center = centroid_quadratic(
        image_data, 
        xpeak = target_x, 
        ypeak = target_y,
        search_boxsize=11, 
        fit_boxsize=5,
    )

    return(supernova_xy_center)

## open_fits

In [ ]:
def open_fits(img_path):
    image = fits.open(img_path)
    img_data = image[0].data
    img_header = image[0].header
    wcs = WCS(img_header)

    clean_data = np.nan_to_num(img_data, nan=0.0)
    
    return(clean_data, wcs)

## get_pixel_scale

In [ ]:
def get_pixel_scale(wcs):
       scale = proj_plane_pixel_scales(wcs.celestial) * 3600
       pixel_scale = scale.mean()
       return pixel_scale

## make_apertures

In [ ]:
def make_apertures(target_coords, aperture_radius, annulus_inner, annulus_outer):
    
    apertures = CircularAperture(
        target_coords, 
        r=aperture_radius
    )
    
    annuli = CircularAnnulus(
        target_coords, 
        r_in=annulus_inner, 
        r_out=annulus_outer
    )

    return(apertures, annuli)

## plot_zoom_section

In [ ]:
def plot_zoom_section(image_data, target_coords, apertures, annuli,
                        object_name, band, group_id, obs_date):

    img_mean, img_sigma = np.mean(image_data), np.std(image_data)
    
    zoom_height = 200
    zoom_width = zoom_height * 1.5
    section_ymin, section_ymax = (target_coords[1] - zoom_height/2), (target_coords[1] + zoom_height/2)
    section_xmin, section_xmax = (target_coords[0] - zoom_width/2), (target_coords[0] + zoom_width/2)
    section = image_data[int(section_ymin):int(section_ymax), int(section_xmin):int(section_xmax)]

    
    plt.imshow(
        image_data,
        vmin = img_mean - 1 * img_sigma,
        vmax =  img_mean + 5 * img_sigma,
        cmap = 'grey',
        origin='lower',
        interpolation='nearest',
    )
    apertures.plot(color = 'blue', lw = .7, alpha = 1);
    annuli.plot(color = 'red', lw = .7, alpha = 1);
    plt.xlabel('X position (pixels)')
    plt.ylabel('Y position (pixels)')
    
    #cbar = plt.colorbar()
    #cbar.set_label('Counts (ADU)')
    plt.title(f'{object_name}  {band}-band  {obs_date}  (group {group_id})')


    plt.xlim(section_xmin,section_xmax)
    plt.ylim(section_ymax,section_ymin)
    #plt.savefig(f'../graphs/{Star_Name}_{Comp_Star_Name}-Zoomed_Image.png', dpi=300, bbox_inches='tight')

    plt.scatter(
        target_coords[0], target_coords[1], 
        s=30, color='red', marker='+', linewidth = 0.5,
        label='Centroid',
    )
    plt.show()

## conduct_photometry

In [ ]:
def conduct_photometry(image_data, aperture, annuli, zpt, zpt_err, gain):
    phot_table = aperture_photometry(image_data, aperture) 
    C_aper = phot_table['aperture_sum'][0]

    sigma_clip = SigmaClip(sigma = 3.0)
    annulus_stats = ApertureStats(image_data, annuli, sigma_clip=sigma_clip)
    sky_median = annulus_stats.median
    sky_std = annulus_stats.std
    
    N_annul = annulus_stats.sum_aper_area.value
    
    #C_annul = phot_table['aperture_sum_1'] 
    #N_annul = annuli.area
    
    N_aper = aperture.area

    C_sky = N_aper * sky_median #(C_annul/N_annul)
    C_obj = C_aper - C_sky

    obj_mag = -2.5 * (np.log10(C_obj)) + zpt

    source_err = C_obj / gain
    sky_err = N_aper * sky_std **2
    sky_level_err = N_aper **2 * sky_std**2/N_annul
    C_obj_err = np.sqrt(source_err + sky_err + sky_level_err)

    inst_mag_err = 1.0857 * C_obj_err / C_obj
    obj_mag_err = np.sqrt(inst_mag_err**2 + zpt_err**2)

    return float(obj_mag), float(obj_mag_err)

## get_errors

In [ ]:
def get_errors():
    return

## get_empty_list

In [ ]:
def get_empty_list(conn):
#    work_sql = """
#        SELECT group_id, object, mosaic_name, zpt_median
#        FROM observations
#        WHERE mosaic_name IS NOT NULL
#            AND zpt_median IS NOT NULL
#            AND mag IS NULL
#            AND error IS NULL
#        ORDER BY group_id
#    """

    work_sql = """
        SELECT group_id, object, telescope, filter, obs_date, num_exps, mosaic_name_f, zpt_median_f, ztp_err_f
        FROM observations
        WHERE mosaic_name_f IS NOT NULL
            AND zpt_median_f IS NOT NULL
            AND mag_f IS NULL
            AND error_f IS NULL
        ORDER BY group_id
    """

    
    work_query = conn.execute(work_sql).fetchall()
    
    #print(len(work_list))

    return(work_query)

## insert_photometry

In [ ]:
def insert_photometry(conn, work_list, mosaic_folder, sn_coordinates,
                      aperture_arcsec, annulus_inner_arcsec, annulus_outer_arcsec,
                      max_centroid_arcsec, show_plots):
#    save_mag_sql = """
#        UPDATE observations
#        SET mag = ?
#        WHERE group_id = ?
#            AND mag IS NULL
#            AND error IS NULL
#    """
#    save_error_sql = """
#        UPDATE observations
#        SET error = ?
#        WHERE group_id = ?
#            AND mag IS NULL
#            AND error IS NULL
#    """


    save_mag_sql = """
        UPDATE observations
        SET mag_f = ?, mag_err_f = ?
        WHERE group_id = ?
            AND mag_f IS NULL
            AND error_f IS NULL
    """
    save_error_sql = """
        UPDATE observations
        SET error_f = ?
        WHERE group_id = ?
            AND mag_f IS NULL
            AND error_f IS NULL
    """




    
    
    mags_saved = 0
    errors_saved = 0
    
    with conn:
        for group_id, object_name, telescope, band, obs_date, num_exps, mosaic_name, zpt, zpt_err in work_list:
     
            short_sn_name = object_name[-4:].lower()
            obj_RA, obj_Dec = sn_coordinates[short_sn_name]
    
            full_mosaic_file = mosaic_folder / mosaic_name
            image_data, wcs = open_fits(full_mosaic_file)

            pixel_scale = get_pixel_scale(wcs)
            aperture_radius = aperture_arcsec / pixel_scale
            annulus_inner = annulus_inner_arcsec / pixel_scale
            annulus_outer = annulus_outer_arcsec / pixel_scale
            
            
            supernova_position = SkyCoord(obj_RA, obj_Dec)
            predicted_x, predicted_y = wcs.world_to_pixel(supernova_position)

            predicted_coords = (float(predicted_x), float(predicted_y))
    
            image_height, image_width = image_data.shape
    
            inside_x = 0 <= predicted_x < image_width
            inside_y = 0 <= predicted_y < image_height
    
            if not (inside_x and inside_y):
                message = "photometry: target outside image"
                cursor = conn.execute(save_error_sql, (message, group_id))
                
                errors_saved = errors_saved + cursor.rowcount
                print(group_id, mosaic_name, " ", message)
                
                continue
    
            target_coords = get_centroid(image_data, wcs, obj_RA, obj_Dec)
    
            centroid_x = target_coords[0]
            centroid_y = target_coords[1]
    
            if not (np.isfinite(centroid_x) and np.isfinite(centroid_y)):
                message = "photometry: no centroid"
                cursor = conn.execute(save_error_sql, (message, group_id))
                
                errors_saved = errors_saved + cursor.rowcount
                print(group_id, mosaic_name, " ", message)

                if show_plots == True:
                    apertures, annuli = make_apertures(predicted_coords, aperture_radius, annulus_inner, annulus_outer)
                    plot_zoom_section(image_data, predicted_coords, apertures, annuli, object_name, band, group_id, obs_date)
                
                continue
    
            
            delta_x = centroid_x - predicted_x
            delta_y = centroid_y - predicted_y
            distance = np.sqrt(delta_x**2 + delta_y**2)
            distance = distance * pixel_scale
    
            if distance > max_centroid_arcsec:
                message = f"photometry: centroid too far ({distance:.1f} arcsec)"
                cursor = conn.execute(save_error_sql, (message, group_id))
                
                errors_saved = errors_saved + cursor.rowcount
                print(group_id, mosaic_name, " ", message)

                if show_plots == True:
                    apertures, annuli = make_apertures(predicted_coords, aperture_radius, annulus_inner, annulus_outer)
                    plot_zoom_section(image_data, predicted_coords, apertures, annuli, object_name, band, group_id, obs_date)
                
                continue




            
            apertures, annuli = make_apertures(target_coords, aperture_radius, annulus_inner, annulus_outer)
    
            if show_plots == True:
                plot_zoom_section(image_data, target_coords, apertures, annuli, object_name, band, group_id, obs_date)
    

            gain = telescope_gain[telescope]
            if num_exps <= 2:
                eff_gain = gain*num_exps
            else:
                eff_gain = gain*num_exps * 2 / np.pi
            
            
            obj_mag, obj_mag_err = conduct_photometry(image_data, apertures, annuli, zpt, zpt_err, eff_gain)

            
            if not (np.isfinite(obj_mag) and np.isfinite(obj_mag_err)):
                message = "photometry: no flux"
                cursor = conn.execute(save_error_sql, (message, group_id))
                
                errors_saved = errors_saved + cursor.rowcount
                print(group_id, mosaic_name, " ", message)
                
                continue
    
            
            cursor = conn.execute(save_mag_sql, (obj_mag, obj_mag_err, group_id))
            mags_saved = mags_saved + cursor.rowcount
    
    return(mags_saved, errors_saved)

# Actual Photometry

In [ ]:
aperture_radius = 12
annulus_inner = 18
annulus_outer = 24
max_centroid_distance = 2      
show_plots = False

work_list = get_empty_list(conn)


aperture_arcsec = 5.0
annulus_inner_arcsec = 15.0
annulus_outer_arcsec = 25.0
max_centroid_arcsec = 2.0
show_plots = True



#print("Mosaics to measure:", len(work_list))

mags_saved, errors_saved = insert_photometry(
    conn, work_list, 
    #nonflat_mosaic_folder,
    flat_mosaic_folder,
    sn_coordinates, aperture_arcsec, annulus_inner_arcsec,
    annulus_outer_arcsec, max_centroid_arcsec, show_plots,
)

print("Number of magnitudes stored:", mags_saved)
print("Number of errors:", errors_saved)

# Forced Photometry

## get_forced_candidates

In [ ]:
def get_forced_candidates(conn):
    candidate_sql = """
        SELECT group_id, object, telescope, filter, obs_date, error_f
        FROM observations
        WHERE mosaic_name_f IS NOT NULL
            AND zpt_median_f IS NOT NULL
            AND mag_f IS NULL
            AND (error_f = 'photometry: no centroid'
                 OR error_f LIKE 'photometry: centroid too far%')
        ORDER BY group_id
    """
    
    for row in conn.execute(candidate_sql):
        print(row)

In [ ]:
get_forced_candidates(conn)

In [ ]:
forced_groups = [1, 2, 41, 84, 86, 124, 136, 137, 141, 142, 143, 145, 146, 152, 153, 169, 173, 179, 195, 203, 208, 209]
#[137, 173, 136, 141, 142, 143, 145, 146, 195, 203, 208, 209]

In [ ]:
def get_forced_row(conn, group_id):
    sql = """
        SELECT group_id, object, telescope, num_exps,
               mosaic_name_f, zpt_median_f, ztp_err_f, error_f
        FROM observations
        WHERE group_id = ?
    """
    return conn.execute(sql, (group_id,)).fetchone()


for group_id in forced_groups:
    row = get_forced_row(conn, group_id)
    group_id, object_name, telescope, num_exps, mosaic_name, zpt, zpt_err, old_error = row

    short_sn_name = object_name[-4:].lower()
    obj_RA, obj_Dec = sn_coordinates[short_sn_name]

    image_data, wcs = open_fits(flat_mosaic_folder / mosaic_name)
    pixel_scale = get_pixel_scale(wcs)

    predicted_x, predicted_y = wcs.world_to_pixel(SkyCoord(obj_RA, obj_Dec))
    target_coords = (float(predicted_x), float(predicted_y))

    apertures, annuli = make_apertures(
        target_coords,
        aperture_arcsec / pixel_scale,
        annulus_inner_arcsec / pixel_scale,
        annulus_outer_arcsec / pixel_scale,
    )

    print(group_id, object_name, telescope, "|", old_error)
    #plot_zoom_section(image_data, target_coords, apertures, annuli)

In [ ]:
def test(conn, forced_groups):
    save_forced_sql = """
        UPDATE observations
        SET mag_f = ?, mag_err_f = ?, forced_f = ?, error_f = NULL
        WHERE group_id = ?
            AND mag_f IS NULL
            AND error_f LIKE 'photometry:%'
    """
    
    forced_saved = 0
    
    with conn:
        for group_id in forced_groups:
            row = get_forced_row(conn, group_id)
            group_id, object_name, telescope, num_exps, mosaic_name, zpt, zpt_err, old_error = row
    
            if old_error is None:
                print(group_id, "has no error any more, skipping")
                continue
    
            short_sn_name = object_name[-4:].lower()
            obj_RA, obj_Dec = sn_coordinates[short_sn_name]
    
            image_data, wcs = open_fits(flat_mosaic_folder / mosaic_name)
            pixel_scale = get_pixel_scale(wcs)
    
            predicted_x, predicted_y = wcs.world_to_pixel(SkyCoord(obj_RA, obj_Dec))
            target_coords = (float(predicted_x), float(predicted_y))
    
            apertures, annuli = make_apertures(
                target_coords,
                aperture_arcsec / pixel_scale,
                annulus_inner_arcsec / pixel_scale,
                annulus_outer_arcsec / pixel_scale,
            )
    
            gain = telescope_gain[telescope]
            if num_exps <= 2:
                eff_gain = gain * num_exps
            else:
                eff_gain = gain * num_exps * 2 / np.pi
    
            obj_mag, obj_mag_err = conduct_photometry(
                image_data, apertures, annuli, zpt, zpt_err, eff_gain
            )
    
            if not (np.isfinite(obj_mag) and np.isfinite(obj_mag_err)):
                print(group_id, "forced flux not positive, left as error")
                continue
    
            cursor = conn.execute(save_forced_sql, (obj_mag, obj_mag_err, old_error, group_id))
            forced_saved = forced_saved + cursor.rowcount
            print(group_id, object_name, f"{obj_mag:.3f} +/- {obj_mag_err:.3f}")
    
    print("Forced magnitudes stored:", forced_saved)